In [ ]:
import sys, torch
print("python:", sys.executable)
print("torch:", torch.__version__)
print("cuda:", torch.version.cuda)
print("avail:", torch.cuda.is_available())

## Data Loading and Initial Inspection

**Purpose:**  
Load the raw CUAD clause dataset (`master_clauses.csv`) and inspect its structure before preprocessing.

**What it does:**  
- Imports required libraries (`os`, `re`, `numpy`, `pandas`) for data handling and preprocessing.  
- Defines the file path for the master clause dataset.  
- Reads the CSV file into a Pandas DataFrame.  
- Prints dataset shape, number of columns, and a preview of the first two rows to understand structure.

**Inputs:**  
- `master_clauses.csv` — raw clause-level dataset extracted from CUAD.

**Outputs:**  
- `df` — a Pandas DataFrame containing all contract records and clause annotations.  
- Printed metadata:
  - Total rows and columns  
  - Sample preview of data  

**Notes / Assumptions:**  
- The CSV file path must be updated if running on a different system.  
- This dataset contains document-level metadata and multiple clause-category columns that will be processed in later steps.  
- At this stage, no cleaning or transformation is performed — this is purely structural inspection.

In [ ]:
# --- Imports ---
import os
import re
import numpy as np
import pandas as pd

# --- Load master_clauses.csv ---
CSV_PATH = r"C:\coding\FL_proj\master_clauses.csv"  # change this

df = pd.read_csv(CSV_PATH)

print("Shape:", df.shape)
print("Columns:", len(df.columns))
print(df.head(2))

## Creating the Clause-Level Dataset

**Purpose:**  
Convert the raw CUAD data into a clean clause-level dataset that contains only real clause examples.

**What it does:**  
- Finds all clause category columns (those ending in `-Answer`).  
- Extracts the category names (e.g., "Anti-Assignment", "Cap On Liability").  
- Defines a helper function to safely read clause spans stored as text lists.  
- Goes through every document and every clause category.  
- Keeps only clauses where the category actually exists (not empty and not "No").  
- Extracts the clause text:
  - Uses span text if available.
  - Otherwise uses the answer text.
- Stores everything in a new DataFrame called `clauses_df`.

**Inputs:**  
- `df` — The raw CUAD dataset.

**Outputs:**  
- `clauses_df` — A dataset where each row represents one real clause.  
  Columns:
  - `doc_filename`
  - `doc_title`
  - `category`
  - `clause_text`

**Notes / Assumptions:**  
- Only real clause examples are kept (these are called "positive samples").  
- A positive sample means the clause type is present in the contract.  
- Clauses marked as `"No"` are ignored.  
- This dataset will later be used to train the clause classification model.

In [ ]:
import ast
import pandas as pd

# 1) Find all label answer columns (these are the 41 categories)
answer_cols = [c for c in df.columns if c.endswith("-Answer")]
categories = [c[:-7] for c in answer_cols]  # remove "-Answer"

print("Num categories:", len(categories))
print("First 10 categories:", categories[:10])

# 2) Helper: parse CUAD list-like strings safely
def parse_span_list(x):
    """
    CUAD span columns often look like:
      "['span1', 'span2']" or "[]"
    Returns a list[str].
    """
    if pd.isna(x):
        return []
    if isinstance(x, list):
        return [s for s in x if isinstance(s, str) and s.strip()]
    if isinstance(x, str):
        s = x.strip()
        if s == "" or s == "[]":
            return []
        try:
            val = ast.literal_eval(s)
            if isinstance(val, list):
                return [t for t in val if isinstance(t, str) and t.strip()]
        except Exception:
            return []
    return []

# 3) Build clause-level dataset (positives only to start)
rows = []
for _, r in df.iterrows():
    doc = r.get("Filename", "")
    doc_title = r.get("Document Name-Answer", "")

    for cat in categories:
        ans = r.get(f"{cat}-Answer", None)

        # positive if answer exists and isn't "No"
        if isinstance(ans, str):
            ans_clean = ans.strip()
        else:
            ans_clean = "" if pd.isna(ans) else str(ans)

        if ans_clean == "" or ans_clean.lower() == "no":
            continue

        spans = parse_span_list(r.get(cat, None))
        # choose clause text: prefer spans, else answer text
        if spans:
            clause_text = " \n\n".join(spans)
        else:
            clause_text = ans_clean

        rows.append({
            "doc_filename": doc,
            "doc_title": doc_title,
            "category": cat,
            "clause_text": clause_text
        })

clauses_df = pd.DataFrame(rows)

print("Clause-level positives:", clauses_df.shape)
print(clauses_df.sample(5, random_state=42))

## Removing Non-Clause Metadata Categories

**Purpose:**  
Remove metadata fields that are not meaningful legal clause types for classification.

**What it does:**  
- Defines a set of categories that represent contract metadata (e.g., document name, dates, parties).  
- Filters the dataset to exclude these categories.  
- Creates a new DataFrame (`clf_df`) that contains only actual legal clause types.  
- Prints the new dataset size and the most common clause categories.

**Inputs:**  
- `clauses_df` — Clause-level dataset containing all extracted clause types.

**Outputs:**  
- `clf_df` — Cleaned dataset containing only meaningful legal clause categories suitable for training.  
- Printed summary:
  - Dataset shape after filtering.
  - Frequency of top clause categories.

**Notes / Assumptions:**  
- Fields like "Document Name" and "Agreement Date" are metadata, not legal obligations or risk-bearing clauses.  
- Removing them improves model quality by ensuring the classifier learns real legal clause types.  
- This step prepares the dataset for supervised clause classification.

In [ ]:
# labels you probably don't want to classify as "clause types"
drop_cats = {
    "Document Name",
    "Parties",
    "Agreement Date",
    "Effective Date",
    "Expiration Date",
}

clf_df = clauses_df[~clauses_df["category"].isin(drop_cats)].copy()

print("After dropping metadata categories:", clf_df.shape)
print("Top categories:\n", clf_df["category"].value_counts().head(15))

## Cleaning Clause Text and Filtering Short Entries

**Purpose:**  
Clean the clause text to ensure consistent formatting and remove very short clauses that may not contain meaningful information.

**What it does:**  
- Defines a `clean_text` function to:
  - Convert input to string.
  - Replace non-breaking spaces with regular spaces.
  - Normalize multiple spaces into a single space.
  - Remove leading and trailing whitespace.
- Applies this cleaning function to the `clause_text` column.
- Removes clauses shorter than 30 characters.
- Prints the dataset shape after cleaning and filtering.

**Inputs:**  
- `clf_df["clause_text"]` — Raw clause text extracted from contracts.

**Outputs:**  
- Updated `clf_df` with cleaned and standardized clause text.
- Dataset with very short clauses removed.

**Notes / Assumptions:**  
- Cleaning improves model performance by removing formatting noise.
- Very short clauses are likely incomplete, noisy, or not informative enough for classification.
- The 30-character threshold can be adjusted later based on experimentation.

In [ ]:
import re

def clean_text(s: str) -> str:
    s = str(s)
    s = s.replace("\u00a0", " ")          # non-breaking space
    s = re.sub(r"\s+", " ", s).strip()    # normalize whitespace
    return s

clf_df["clause_text"] = clf_df["clause_text"].map(clean_text)

# drop very short clauses (tune threshold later)
clf_df = clf_df[clf_df["clause_text"].str.len() >= 30].copy()

print("After cleaning + length filter:", clf_df.shape)

## Train–Test Split by Contract (Group-Based Splitting)

**Purpose:**  
Split the dataset into training and testing sets while ensuring that clauses from the same contract do not appear in both sets.

**What it does:**  
- Uses `GroupShuffleSplit` from scikit-learn to perform a grouped split.
- Treats each `doc_filename` (contract) as a group.
- Splits the data into:
  - 80% training set
  - 20% testing set
- Ensures that all clauses from a single contract go entirely into either train or test.
- Prints dataset sizes and number of unique contracts in each split.

**Inputs:**  
- `clf_df` — Cleaned clause-level dataset.
- `doc_filename` — Used as grouping key to prevent data leakage.

**Outputs:**  
- `train_df` — Training dataset.
- `test_df` — Testing dataset.
- Printed summary of dataset sizes and contract counts.

**Notes / Assumptions:**  
- Group-based splitting prevents data leakage, which would occur if clauses from the same contract were in both train and test sets.
- This ensures a more realistic evaluation, since the model is tested on entirely unseen contracts.
- Random seed (`random_state=42`) ensures reproducibility.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

groups = clf_df["doc_filename"]
train_idx, test_idx = next(gss.split(clf_df, groups=groups))

train_df = clf_df.iloc[train_idx].reset_index(drop=True)
test_df  = clf_df.iloc[test_idx].reset_index(drop=True)

print("Train:", train_df.shape, "Test:", test_df.shape)
print("Unique contracts - train:", train_df["doc_filename"].nunique(),
      "test:", test_df["doc_filename"].nunique())

## Encoding Clause Categories into Numeric Labels

**Purpose:**  
Convert textual clause categories into numeric labels required for training the classification model.

**What it does:**  
- Extracts the unique clause categories from the training dataset.
- Sorts them for consistency.
- Creates:
  - `label2id` — maps each category name to a numeric ID.
  - `id2label` — maps each numeric ID back to the category name.
- Adds a new `label` column to both `train_df` and `test_df` using the numeric encoding.
- Prints the total number of labels and a sample of category names.

**Inputs:**  
- `train_df["category"]`
- `test_df["category"]`

**Outputs:**  
- `label2id` dictionary (category → number)
- `id2label` dictionary (number → category)
- New `label` column in both train and test datasets

**Notes / Assumptions:**  
- Machine learning models require numeric labels rather than text.
- The label mapping is built using only the training data to maintain consistency.
- `id2label` is later used for interpreting model predictions.
- Sorting ensures reproducibility of label indexing.

In [ ]:
labels = sorted(train_df["category"].unique())
label2id = {l:i for i,l in enumerate(labels)}
id2label = {i:l for l,i in label2id.items()}

train_df["label"] = train_df["category"].map(label2id)
test_df["label"] = test_df["category"].map(label2id)

print("Num labels:", len(labels))
print(labels[:15])

## Inspecting Test Set Categories

**Purpose:**  
Verify which clause categories are present in the test dataset.

**What it does:**  
- Prints all unique clause categories in `test_df`.
- Allows inspection of label distribution in the test split.

**Inputs:**  
- `test_df["category"]` — Clause category column from the test dataset.

**Outputs:**  
- List of unique clause categories present in the test set.

**Notes / Assumptions:**  
- This step helps confirm that the test set contains valid and expected categories.
- Since the label mapping was built from training data, all test categories should already exist in the training label set.
- This is mainly a sanity check before model training.

In [ ]:
print(test_df["category"].unique())

## Verifying PyTorch and GPU Configuration

**Purpose:**  
Confirm that PyTorch is correctly installed and that GPU (CUDA) acceleration is available for training.

**What it does:**  
- Prints the installed PyTorch version.
- Displays the CUDA version linked with PyTorch.
- Checks whether CUDA is available.
- Prints the name of the detected GPU device.

**Inputs:**  
- Local Python environment with PyTorch installed.

**Outputs:**  
- PyTorch version number.
- CUDA version (if available).
- Boolean indicating GPU availability.
- GPU device name.

**Notes / Assumptions:**  
- GPU acceleration significantly speeds up transformer model training.
- If `torch.cuda.is_available()` returns `False`, the model will run on CPU instead.
- This step ensures the environment is correctly configured before training begins.

In [ ]:
import torch
print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

## Training the LegalBERT Clause Classification Model

**Purpose:**  
Train a transformer-based classifier (LegalBERT) to predict the clause type for each clause in the CUAD dataset.

**What it does:**  
- Imports the libraries needed for training (HuggingFace `datasets` + `transformers`, PyTorch, and metrics).  
- Converts the Pandas train/test DataFrames into HuggingFace `Dataset` objects so they can be efficiently tokenized and fed into a model.  
- Loads the LegalBERT tokenizer and tokenizes clause text (truncates long clauses to 256 tokens).  
- Uses dynamic padding (`DataCollatorWithPadding`) so each batch is padded only as much as needed (more memory efficient).  
- Loads a LegalBERT model for multi-class classification with `num_labels = number of clause categories`.  
- Defines evaluation metrics (accuracy, macro F1, weighted F1).  
- Sets training configuration (epochs, learning rate, batch size, evaluation schedule, GPU usage).  
- Trains the model and evaluates it on the test set after training.  
- Saves the best model and tokenizer for later inference.

**Inputs:**  
- `train_df` with columns: `["clause_text", "label"]`  
- `test_df` with columns: `["clause_text", "label"]`  
- `label2id` and `id2label` mappings (category ↔ integer label)

**Outputs:**  
- Fine-tuned LegalBERT classifier stored in:
  - `./legalbert_cuad_clauseclf/best_model`
- Console output:
  - Training progress (loss, accuracy, F1)
  - Final evaluation metrics

**Notes / Assumptions:**  
- The model is trained as a **multi-class, single-label classifier** (one clause → one category).  
- Clauses longer than 256 tokens are truncated to fit the model input length.  
- `fp16=torch.cuda.is_available()` enables faster training on GPU when CUDA is available.  
- Dynamic padding is important to reduce GPU memory usage compared to padding all clauses to a fixed maximum length.  
- Saving both the model and tokenizer is required to run inference later in the risk scoring pipeline.

**Important lines (core logic):**  
- `train_ds = Dataset.from_pandas(...)` and `test_ds = Dataset.from_pandas(...)` — converts Pandas → HuggingFace datasets.  
- `train_ds = train_ds.map(tokenize, ...)` — tokenizes text into model-ready inputs.  
- `model = AutoModelForSequenceClassification.from_pretrained(...)` — loads LegalBERT for classification with correct label mappings.  
- `trainer.train()` — runs fine-tuning.  
- `trainer.save_model(save_dir)` + `tokenizer.save_pretrained(save_dir)` — saves the trained model for inference.

In [ ]:
# =========================
# LegalBERT clause classifier training (working)
# - No Trainer(tokenizer=...) arg (new Transformers API)
# - Uses dynamic padding via DataCollatorWithPadding
# - GPU + fp16 automatically if available
# =========================

import re
import numpy as np
import torch

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)
from sklearn.metrics import accuracy_score, f1_score

# ---- ASSUMES YOU ALREADY HAVE THESE FROM YOUR PREVIOUS STEPS ----
# train_df: columns ["clause_text", "label"]
# test_df : columns ["clause_text", "label"]
# label2id: dict label->id
# id2label: dict id->label
# ---------------------------------------------------------------

MODEL_NAME = "nlpaueb/legal-bert-base-uncased"

# Optional: basic cleanup (safe to run even if already cleaned)
def clean_text(s: str) -> str:
    s = str(s).replace("\u00a0", " ")
    s = re.sub(r"\s+", " ", s).strip()
    return s

train_df = train_df.copy()
test_df = test_df.copy()
train_df["clause_text"] = train_df["clause_text"].map(clean_text)
test_df["clause_text"] = test_df["clause_text"].map(clean_text)

# Build HF datasets
train_ds = Dataset.from_pandas(train_df[["clause_text", "label"]], preserve_index=False)
test_ds  = Dataset.from_pandas(test_df[["clause_text", "label"]], preserve_index=False)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(batch["clause_text"], truncation=True, max_length=256)

train_ds = train_ds.map(tokenize, batched=True, desc="Tokenizing train")
test_ds  = test_ds.map(tokenize, batched=True, desc="Tokenizing test")

# Dynamic padding (better memory use than padding to max_length everywhere)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Model
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id,
)

# Metrics
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
        "f1_weighted": f1_score(labels, preds, average="weighted"),
    }

# Training args
# NOTE: some versions use "evaluation_strategy" not "eval_strategy"
args = TrainingArguments(
    output_dir="./legalbert_cuad_clauseclf",
    eval_strategy="epoch",          # <-- use this
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    num_train_epochs=3,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    weight_decay=0.01,
    logging_steps=50,
    fp16=torch.cuda.is_available(),
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()
print("Eval:", trainer.evaluate())

# Save best model + tokenizer
save_dir = "./legalbert_cuad_clauseclf/best_model"
trainer.save_model(save_dir)
tokenizer.save_pretrained(save_dir)
print("Saved to:", save_dir)

## Evaluating the Clause Classification Model

**Purpose:**  
Evaluate the trained LegalBERT model on the test dataset and generate detailed classification metrics.

**What it does:**  
- Uses `trainer.predict(test_ds)` to obtain model predictions on the test set.  
- Extracts:
  - `y_true` — the actual labels from the dataset.
  - `y_pred` — predicted labels (computed by taking the class with highest probability).  
- Converts numeric label IDs back to readable category names using `id2label`.  
- Generates a detailed classification report including:
  - Precision
  - Recall
  - F1-score
  - Support (number of samples per class)

**Inputs:**  
- `trainer` — trained LegalBERT model.
- `test_ds` — tokenized test dataset.
- `id2label` — mapping from numeric labels to category names.

**Outputs:**  
- Printed classification report showing performance per clause category.

**Notes / Assumptions:**  
- `np.argmax(pred.predictions, axis=1)` selects the most probable class for each clause.  
- The classification report helps identify:
  - Which clause types perform well.
  - Which rare categories may suffer from low recall or precision.
- Macro F1 is useful for understanding performance across all classes equally.
- Weighted F1 accounts for class imbalance.

**Important lines:**  
- `trainer.predict(test_ds)` → runs inference on the test set.  
- `np.argmax(...)` → converts probabilities into predicted class IDs.  
- `classification_report(...)` → prints detailed performance metrics per category.

In [ ]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

pred = trainer.predict(test_ds)
y_true = pred.label_ids
y_pred = np.argmax(pred.predictions, axis=1)

# id -> label name
labels_sorted = [id2label[i] for i in range(len(id2label))]

print(classification_report(y_true, y_pred, target_names=labels_sorted, digits=3))

## Risk Scoring Pipeline Setup (Run Once)

**Purpose:**  
Set up everything needed to score clause risk: load the trained clause classifier, load a sentence embedding model for similarity scoring, and define the keyword/rule resources used to compute risk.

**What it does:**  
- Imports required libraries for:
  - text processing (`re`, `math`)
  - data handling (`numpy`, `pandas`)
  - GPU / model inference (`torch`)
  - transformer models (`transformers`)
  - semantic similarity (`sentence_transformers`, `cosine_similarity`)
- Loads the fine-tuned LegalBERT clause classifier from disk and moves it to GPU if available.
- Defines `predict_clause_type_and_confidence()` which:
  - tokenizes a clause
  - runs the classifier
  - returns the predicted clause category + confidence score
- Loads a SentenceTransformer embedding model to compute similarity between a clause and "high-risk exemplar" texts.
- Defines:
  - `RISK_LEXICON`: regex patterns for risky words/phrases with weights (keyword-based risk).
  - `HIGH_RISK_EXEMPLARS`: example high-risk clauses per clause type (used for similarity risk).

**Inputs:**  
- Saved clause classification model directory: `./legalbert_cuad_clauseclf/best_model`  
- SentenceTransformer model name: `"sentence-transformers/all-MiniLM-L6-v2"`

**Outputs:**  
- `clf_tokenizer` and `clf_model` — trained clause classifier loaded into memory.  
- `predict_clause_type_and_confidence()` — helper function for inference.  
- `emb_model` — sentence embedding model used for similarity scoring.  
- `RISK_LEXICON` — keyword dictionary used for keyword-based risk scoring.  
- `HIGH_RISK_EXEMPLARS` — exemplar library used for similarity-based risk scoring.

**Notes / Assumptions:**  
- This cell should be run once per session (it loads models into memory).  
- `device = "cuda" ...` enables GPU acceleration if CUDA is available; otherwise it falls back to CPU.  
- The clause classifier predicts the clause **type**, which later drives rule-based scoring and exemplar selection.  
- The lexicon and exemplars are manually defined and should be expanded/tuned for better risk scoring coverage.

**Important lines (must understand):**  
- `clf_model = AutoModelForSequenceClassification.from_pretrained(...).to(device).eval()`  
  → loads the trained classifier, moves it to GPU/CPU, and sets inference mode.  
- `@torch.no_grad()`  
  → disables gradient tracking to make prediction faster and save memory.  
- `emb_model = SentenceTransformer(..., device=device)`  
  → loads the sentence embedding model for similarity scoring.  
- `RISK_LEXICON = {...}` and `HIGH_RISK_EXEMPLARS = {...}`  
  → these define the "knowledge" used for keyword and similarity-based risk scoring.

In [ ]:
# =========================
# SETUP (run once)
# =========================
import re, math
import numpy as np
import pandas as pd
import torch

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# 0) Load your trained classifier
CLAUSE_CLF_DIR = "./legalbert_cuad_clauseclf/best_model"  # change if needed
device = "cuda" if torch.cuda.is_available() else "cpu"

clf_tokenizer = AutoTokenizer.from_pretrained(CLAUSE_CLF_DIR)
clf_model = AutoModelForSequenceClassification.from_pretrained(CLAUSE_CLF_DIR).to(device).eval()

@torch.no_grad()
def predict_clause_type_and_confidence(text: str, max_len: int = 256):
    inputs = clf_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=max_len,
    ).to(device)

    logits = clf_model(**inputs).logits
    probs = torch.softmax(logits, dim=-1).squeeze(0)
    pred_id = int(torch.argmax(probs).item())
    conf = float(probs[pred_id].item())

    id2label = clf_model.config.id2label
    pred_label = id2label[str(pred_id)] if isinstance(id2label, dict) and str(pred_id) in id2label else id2label[pred_id]
    return pred_label, conf

# 1) Sentence Transformer model (for similarity scoring)
EMB_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
emb_model = SentenceTransformer(EMB_MODEL_NAME, device=device)

# 2) Keyword lexicon (risk words/phrases)
RISK_LEXICON = {
    r"\bunlimited\b": 3.0,
    r"\bwithout\s+limitation\b": 2.5,
    r"\bsole\s+discretion\b": 2.5,
    r"\bat\s+any\s+time\b": 1.5,
    r"\bwithout\s+cause\b": 2.0,
    r"\bwithout\s+notice\b": 2.0,
    r"\bimmediately\b": 1.5,
    r"\birrevocable\b": 2.0,
    r"\bperpetual\b": 2.0,
    r"\bin\s+perpetuity\b": 2.0,
    r"\bnon[-\s]?refundable\b": 2.0,
    r"\bpenalt(y|ies)\b": 2.0,
    r"\bliquidated\s+damages\b": 2.0,
    r"\bindemnif(y|ies|ication)\b": 2.0,
    r"\bhold\s+harmless\b": 1.5,
    r"\bconsequential\s+damages\b": 1.5,
    r"\bpunitive\s+damages\b": 1.5,
    r"\battorneys['']?\s+fees\b": 1.0,
    r"\binjunctive\s+relief\b": 1.0,
    r"\baudit\b": 0.75,
    r"\bassign(ment|)\b": 0.5,
    r"\bnon[-\s]?compete\b": 2.0,
    r"\bexclusive\b": 1.0,
    r"\bminimum\s+commitment\b": 1.0,
}

# 3) High-risk exemplars (expand later)
HIGH_RISK_EXEMPLARS = {
    "Cap On Liability": [
        "Vendor's total liability is unlimited and includes consequential and punitive damages.",
        "Customer may recover all damages without limitation and without any cap."
    ],
    "Termination For Convenience": [
        "Either party may terminate this agreement for convenience at any time without cause and without notice.",
        "Company may terminate immediately at its sole discretion."
    ],
    "Non-Compete": [
        "The other party shall not compete worldwide for a period of five years after termination.",
        "Licensee agrees to a broad non-compete covering all competing products."
    ],
    "Liquidated Damages": [
        "In the event of breach, party shall pay liquidated damages as a penalty in addition to other remedies."
    ],
    "Anti-Assignment": [
        "Any assignment without prior written consent is void and constitutes a material breach."
    ],
    "Exclusivity": [
        "Supplier is the exclusive provider and customer may not purchase from any other supplier."
    ],
}

## Explainable Risk Scoring + Ranking

This block implements three complementary risk scoring methods:
1. **Keyword-based scoring** - detects risky terminology
2. **Rule-based scoring** - applies clause-type specific heuristics  
3. **Similarity-based scoring** - compares to high-risk exemplars

All scores are combined into a final explainable risk assessment (0-10).

In [ ]:
import re
import math
import numpy as np
import pandas as pd
import torch
from sklearn.metrics.pairwise import cosine_similarity

def keyword_risk_explain(text: str):
    t = text.lower()
    raw = 0.0
    hits = []
    for pattern, w in RISK_LEXICON.items():
        found = re.findall(pattern, t)
        c = len(found)
        if c:
            capped = min(c, 3)
            contrib = w * capped
            raw += contrib
            hits.append({
                "pattern": pattern,
                "count": c,
                "weight": w,
                "contribution": contrib
            })
    score = 10.0 * (1 - math.exp(-raw / 4.0))
    score = float(max(0.0, min(10.0, score)))
    hits.sort(key=lambda d: d["contribution"], reverse=True)
    return score, hits

def _re(text: str, pat: str) -> bool:
    return re.search(pat, text.lower()) is not None

def rule_risk_explain(text: str, clause_type: str):
    t = text.lower()
    ct = (clause_type or "").lower()
    score = 0.0
    reasons = []

    if _re(t, r"\bunlimited\b") or _re(t, r"\bwithout\s+limitation\b"):
        score += 3.0
        reasons.append("Mentions unlimited/without limitation (+3.0)")
    if _re(t, r"\bsole\s+discretion\b"):
        score += 2.0
        reasons.append("Mentions sole discretion (+2.0)")
    if _re(t, r"\bwithout\s+notice\b") or _re(t, r"\bimmediately\b"):
        score += 2.0
        reasons.append("Mentions without notice/immediately (+2.0)")

    if "cap on liability" in ct:
        if _re(t, r"\bfees\s+paid\b") or _re(t, r"\bamounts?\s+paid\b") or _re(t, r"\bpaid\s+under\s+this\s+agreement\b"):
            score += 3.5
            reasons.append("Liability capped to fees/amounts paid (+3.5)")
    elif "termination for convenience" in ct:
        if _re(t, r"\bwithout\s+cause\b") or _re(t, r"\bfor\s+convenience\b") or _re(t, r"\bat\s+any\s+time\b"):
            score += 5.0
            reasons.append("Termination for convenience/without cause (+5.0)")
    elif "non-compete" in ct:
        score += 6.0
        reasons.append("Non-compete clause (+6.0)")

    score = float(max(0.0, min(10.0, score)))
    return score, reasons

_exemplar_cache = {}

def _get_exemplar_embeddings(clause_type: str):
    if clause_type not in _exemplar_cache:
        ex = HIGH_RISK_EXEMPLARS.get(clause_type, [])
        if not ex:
            _exemplar_cache[clause_type] = None
        else:
            _exemplar_cache[clause_type] = emb_model.encode(ex, normalize_embeddings=True)
    return _exemplar_cache[clause_type]

def similarity_risk_explain(text: str, clause_type: str):
    exemplars = HIGH_RISK_EXEMPLARS.get(clause_type, [])
    ex_emb = _get_exemplar_embeddings(clause_type)
    if ex_emb is None:
        return 0.0, {"best_exemplar": None, "cosine": None}

    q = emb_model.encode([text], normalize_embeddings=True)
    sims = cosine_similarity(q, ex_emb)[0]
    best_idx = int(np.argmax(sims))
    best_sim = float(sims[best_idx])
    sim_score = float(max(0.0, min(10.0, 10.0 * best_sim)))

    return sim_score, {"best_exemplar": exemplars[best_idx], "cosine": best_sim}

def final_risk_score(keyword_s: float, rule_s: float, sim_s: float,
                     w_keyword=0.35, w_rule=0.45, w_sim=0.20) -> float:
    return float(max(0.0, min(10.0, (w_keyword*keyword_s) + (w_rule*rule_s) + (w_sim*sim_s))))

def score_and_rank_clauses_explainable(
    clauses: pd.DataFrame,
    text_col: str = "clause_text",
    group_col: str = "doc_filename",
    top_k_per_doc: int = 10
) -> pd.DataFrame:
    df = clauses.copy()
    df[text_col] = df[text_col].astype(str).str.replace(r"\s+", " ", regex=True).str.strip()

    preds = df[text_col].apply(predict_clause_type_and_confidence)
    df["pred_clause_type"] = preds.apply(lambda x: x[0])
    df["pred_confidence"] = preds.apply(lambda x: x[1])

    kw_scores, rule_scores, sim_scores, final_scores = [], [], [], []

    for text, ct, conf in zip(df[text_col].tolist(), df["pred_clause_type"].tolist(), df["pred_confidence"].tolist()):
        kw_s, _ = keyword_risk_explain(text)
        rule_s, _ = rule_risk_explain(text, ct)
        sim_s, _ = similarity_risk_explain(text, ct)
        final_s = final_risk_score(kw_s, rule_s, sim_s)

        kw_scores.append(kw_s)
        rule_scores.append(rule_s)
        sim_scores.append(sim_s)
        final_scores.append(final_s)

    df["risk_keyword"] = kw_scores
    df["risk_rule"] = rule_scores
    df["risk_similarity"] = sim_scores
    df["risk_final"] = final_scores

    if group_col in df.columns:
        df["risk_rank_in_doc"] = df.groupby(group_col)["risk_final"].rank(method="first", ascending=False).astype(int)
        ranked = df.sort_values([group_col, "risk_final"], ascending=[True, False])
    else:
        ranked = df.sort_values("risk_final", ascending=False).reset_index(drop=True)

    return ranked.reset_index(drop=True)

## Export Risk Scores to JSONL

Export the scored clauses to a client-ready JSONL format for frontend integration.

In [ ]:
import json
import numpy as np
import pandas as pd

def to_serializable(obj):
    if obj is None:
        return None
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, (np.ndarray,)):
        return obj.tolist()
    return obj

# Example usage - replace with your actual data
demo = pd.DataFrame({
    "doc_filename": ["DocA", "DocA", "DocB"],
    "clause_text": [
        "Either party may terminate this Agreement for convenience at any time without cause and without notice.",
        "In no event shall liability exceed fees paid under this Agreement.",
        "Any assignment without the prior written consent of Company is void."
    ]
})

ranked = score_and_rank_clauses_explainable(demo, top_k_per_doc=10)

OUT_PATH = "client_risk_scores.jsonl"
with open(OUT_PATH, "w", encoding="utf-8") as f:
    for record in ranked.to_dict(orient="records"):
        clean_record = {k: to_serializable(v) for k, v in record.items()}
        f.write(json.dumps(clean_record, ensure_ascii=False) + "\n")

print(f"Saved {len(ranked)} records to:", OUT_PATH)